# 메타데이터 조건화 (FiLM) + 주기 인코딩 — 회전 숫자 (Colab)

**연습 기법** (IOAI 2025 *at-home* **Weather**(위성 강수 분할) 와 동일): 모델에 **영상 외 메타데이터**를 녹여
정확도를 올린다. Weather 모범답안의 핵심은 `HybridSolarElevationDataset` — 위경도·UTC 시각에서 **태양 고도각**
(물리식)을 계산하고 시각·각도를 **sin/cos 주기 인코딩**한 뒤, 그 메타데이터로 U-Net 특징을 **조건화(FiLM)**
한다(`m = m * a`). 즉 *이미지 + 물리/메타 정보 융합*.

**대회**: [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) (MNIST). 숫자를 **무작위 각도로 회전**시키면
(Weather 의 태양 각도 같은 *물리적 취득 변수* 모사) **6↔9 가 헷갈려** 이미지만으로는 한계가 생긴다. 이때
**회전 각도**를 주기 인코딩해 **FiLM** 으로 분류기에 주입하면 모호함이 풀린다 — Weather 에서 태양 고도가
비 탐지를 돕는 것과 **동형**이며, *메타데이터가 실제로 필요한* 상황이라 효과가 크고 안정적이다.

**핵심 흐름 & 배울 점**:
1. **주기 인코딩**(sin/cos) — Weather 가 시각·각도 등 주기 변수에 쓴 기법. (각도 0°↔360° 가 인접)
2. **FiLM 조건화** — 메타 벡터 → (γ, β) → CNN 특징 `h·(1+γ)+β`. (Weather 의 `m*a` 일반화)
3. **이미지만 vs 메타+FiLM** 비교 — 회전 모호성(6/9)을 각도 메타가 풀어 정확도 ↑ (≈ +7%p).
4. 실제 테스트(정자세)는 각도 0 으로 추론해 정상 제출.

> ⚙️ **GPU 권장**: 작은 CNN 2회 학습이라 T4 에서 ~2분. (08·13·15 와 같은 digit-recognizer)
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.
> 💡 메타데이터를 **안 쓰면** 회전 6/9 를 원리상 구분 못 함 — 그래서 메타 융합이 본질적으로 필요한 예제.

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scipy", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 Digit Recognizer 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 주기 인코딩 (Weather 의 sin/cos 기법)
각도·시각처럼 **주기적인** 변수는 그냥 숫자로 넣으면 0°와 360°가 멀어 보인다. `sin/cos` 로 인코딩하면
원형으로 이어져 모델이 다루기 쉽다. Weather 는 태양 고도각·시각·월에 이 기법을 썼다.

In [ ]:
import numpy as np
def cyclic_encode(value, period):
    """주기 변수 → (sin, cos). 예: 각도(period=360), 시각(24), 월(12)."""
    theta = 2 * np.pi * (np.asarray(value) / period)
    return np.stack([np.sin(theta), np.cos(theta)], axis=-1)

for deg in [0, 90, 180, 350]:
    s, c = cyclic_encode(deg, 360)
    print(f"{deg:3d}° → sin {s:+.2f}, cos {c:+.2f}")
print("→ 350°와 0°가 (sin,cos) 공간에서 인접 — 주기성이 보존됨.")

## 3. 데이터 로드 & 무작위 회전 + 각도 메타데이터
각 숫자를 0~360° 무작위 회전(Weather 의 *물리적 취득 변수* 모사)하고, 그 **회전 각도**를 주기 인코딩해
메타데이터로 둔다. 회전하면 6/9 등이 모호해져, 이미지만으로는 구분 한계가 생긴다.

In [ ]:
import pandas as pd
from scipy.ndimage import rotate

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
N = 20000                                   # 속도용 서브샘플(전체 42k 중)
train = train.sample(N, random_state=0).reset_index(drop=True)
y = train["label"].values.astype("int64")
X0 = train.drop(columns="label").values.astype("float32").reshape(-1, 28, 28) / 255.0

rng = np.random.RandomState(100)
angles = rng.uniform(0, 360, len(X0)).astype("float32")
Xr = np.stack([rotate(X0[i], angles[i], reshape=False, order=1, mode="constant")
               for i in range(len(X0))]).astype("float32")
meta = cyclic_encode(angles, 360).astype("float32")     # (N, 2)
print("회전 이미지:", Xr.shape, "| 각도 메타:", meta.shape)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 6, figsize=(12, 2.4))
for k in range(6):
    ax[k].imshow(Xr[k], cmap="gray"); ax[k].set_title(f"y={y[k]}, {angles[k]:.0f}°"); ax[k].axis("off")
plt.suptitle("무작위 회전된 숫자 — 각도를 모르면 6/9 등이 모호"); plt.show()

## 4. FiLM 조건화 CNN 정의
메타 벡터(각도 sin/cos) → MLP → **(γ, β)** 로 특징맵을 `h·(1+γ)+β` 변조. `mdim=0` 이면 순수 이미지 CNN.

In [ ]:
import torch, torch.nn as nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FiLMCNN(nn.Module):
    def __init__(self, mdim=0):
        super().__init__(); self.mdim = mdim
        self.feat = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )                                            # → (32, 7, 7)
        if mdim > 0:
            self.film = nn.Sequential(nn.Linear(mdim, 32), nn.ReLU(inplace=True), nn.Linear(32, 64))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(inplace=True), nn.Linear(128, 10))
    def forward(self, x, m=None):
        h = self.feat(x)
        if self.mdim > 0 and m is not None:          # FiLM: 메타로 특징 변조
            gb = self.film(m); g, b = gb[:, :32], gb[:, 32:]
            h = h * (1 + g.view(-1, 32, 1, 1)) + b.view(-1, 32, 1, 1)
        return self.head(h)

print("FiLM-CNN 파라미터:", sum(p.numel() for p in FiLMCNN(mdim=2).parameters()))

## 5. 학습 — 이미지만 vs 메타+FiLM (동일 조건)
`use_meta` 만 바꿔 각도 메타 조건화의 효과를 분리한다.

In [ ]:
Xt = torch.from_numpy(Xr)[:, None]; At = torch.from_numpy(meta); Yt = torch.from_numpy(y)
idx = np.random.RandomState(1).permutation(len(Xr)); cut = int(len(Xr) * 0.85)
tr_i, va_i = idx[:cut], idx[cut:]
EPOCHS = 6; BS = 128

def train_cnn(use_meta, seed=0):
    torch.manual_seed(seed)
    net = FiLMCNN(mdim=2 if use_meta else 0).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3); ce = nn.CrossEntropyLoss()
    for epoch in range(EPOCHS):
        net.train(); perm = np.random.RandomState(epoch).permutation(tr_i)
        for i in range(0, len(perm), BS):
            b = perm[i:i+BS]; mb = At[b].to(device) if use_meta else None
            opt.zero_grad(); ce(net(Xt[b].to(device), mb), Yt[b].to(device)).backward(); opt.step()
    net.eval()
    with torch.no_grad():
        mb = At[va_i].to(device) if use_meta else None
        pred = net(Xt[va_i].to(device), mb).argmax(1).cpu().numpy()
    return net, float((pred == y[va_i]).mean())

_,     acc_img  = train_cnn(use_meta=False)
model, acc_meta = train_cnn(use_meta=True)
print(f"이미지만        val acc = {acc_img:.4f}")
print(f"각도 메타+FiLM  val acc = {acc_meta:.4f}   (Δ {acc_meta - acc_img:+.4f})")
print("\n→ 회전 각도(주기 인코딩)를 FiLM 으로 주입해 6/9 모호성을 해소 — Weather 의 메타 조건화와 동형.")

## 6. 캐글 제출 — 실제 테스트(정자세)는 각도 0 으로 추론
실제 `test.csv` 는 회전되지 않은 정자세이므로 **각도=0** 메타(`sin0,cos0 = 0,1`)로 FiLM 모델을 추론한다.

In [ ]:
test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
Xte = test.values.astype("float32").reshape(-1, 28, 28) / 255.0
Xte_t = torch.from_numpy(Xte)[:, None]
m0 = torch.from_numpy(cyclic_encode(np.zeros(len(Xte)), 360).astype("float32"))   # 각도 0

model.eval(); preds = []
with torch.no_grad():
    for i in range(0, len(Xte_t), 512):
        preds.append(model(Xte_t[i:i+512].to(device), m0[i:i+512].to(device)).argmax(1).cpu().numpy())
test_pred = np.concatenate(preds)

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"ImageId": np.arange(1, len(test_pred) + 1), "Label": test_pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(test_pred))
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) 에 제출하세요.

**기법 요약** (IOAI 2025 Weather): 모델에 **영상 외 메타데이터**를 **FiLM** 으로 조건화 — `메타→(γ,β)→` 특징
`h·(1+γ)+β`. 주기 변수(각도·시각)는 **sin/cos** 로 인코딩(0°↔360° 인접). 메타가 본질적으로 필요한 회전 6/9
문제에서 정확도가 +7%p 안정적으로 오른다 — Weather 가 태양 고도/시각 메타로 비 탐지를 돕는 것과 동형.
더 끌어올리려면: FiLM 을 여러 층에 적용, 메타에 위치/시간/물리 feature 추가, 학습 때 정자세도 섞어 실제 제출 강화.